## Milestone -2 

In [ ]:
# ==============================
# Milestone 2: Email Assistant Evaluation
# ==============================

import pandas as pd

# Load your updated CSV with ground truth
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,urgent
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,neutral


In [ ]:
# ==============================
# Define a simple rule-based email assistant
# ==============================


def email_assistant(email_text):
    text = email_text.lower() # convert text to lowercase for consistency

    # Urgent cues
    if "urgent" in text or "deadline" in text:
        return "notify", "urgent"
    
     # Polite cues
    elif "thank you" in text or "thanks" in text:
        return "ignore", "polite"
    
     # Default action and tone
    else:
        return "respond", "neutral"


In [ ]:
# ==============================
# Apply the email assistant to each email
# ==============================


predictions = []

# Loop through each email and generate predictions
for _, row in df.iterrows():
    action, tone = email_assistant(row["body"])
    predictions.append({
        "id": row["id"],
        "predicted_intent": action,
        "predicted_tone": tone
    })


#Convert the list of predictions into a DataFrame
pred_df = pd.DataFrame(predictions)

# Preview predictions
pred_df.head()


,id,predicted_intent,predicted_tone
0,1,respond,neutral
1,2,respond,neutral
2,3,respond,neutral
3,4,respond,neutral
4,5,respond,neutral


In [ ]:
# ==============================
# Evaluate predictions against ground truth
# ==============================

def evaluate(row):
    score = 0
    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1
    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1
    return score



# Merge predictions with the original dataset
eval_df = df.merge(pred_df, on="id")

# Apply evaluation function row by row
eval_df["score"] = eval_df.apply(evaluate, axis=1)


# ==============================
# Calculate overall accuracy
# ==============================

# Accuracy formula: sum of correct predictions / total possible points
accuracy = (eval_df["score"].sum() / (len(eval_df) * 2)) * 100
print("Overall Accuracy:", accuracy)


Overall Accuracy: 52.5


In [ ]:
# ==============================
# Save the final evaluation output
# ==============================

# Save the merged dataset with predictions and scores

eval_df.to_csv("../data/milestone2_output_Ayesha.csv", index=False)